# 01 Normalize STVR license addresses

Applies USPS Publication 28 uppercase/spacing conventions, positional directionals, recognized street suffixes and secondary designators. This is **format normalization, not USPS/CASS delivery validation**; no ZIP lookup, geocoding, typo correction, or parcel/CAD matching occurs. The supported suffix dictionary is intentionally explicit; unsupported forms are flagged rather than guessed. City/state/ZIP parsing currently supports the supplied Savannah, Georgia format only; other localities require extension.

References: [Pub 28 ?211](https://pe.usps.com/text/pub28/28c2_001.htm), [?233 directionals](https://pe.usps.com/text/pub28/28c2_014.htm), [C1 suffixes](https://pe.usps.com/text/pub28/28apc_002.htm), [?213 secondary units](https://pe.usps.com/text/pub28/28c2_003.htm), [C2 unit designators](https://pe.usps.com/text/pub28/28apc_003.htm).

Separate unit identifiers are retained. A bare identifier receives `#` (unknown designator), never an invented APT designation. Descriptive notes and alternate addresses remain in original fields and enter review. Conflicting units are not resolved automatically; unit match keys are blank for review rows. Building keys deliberately omit units for later parcel candidate generation, not confirmed matches. Street spelling, house-number hyphens/fractions, and unit hyphens are preserved; `6A` and `6-A` are not silently equated. All licenses remain separate, including duplicate addresses and expired licenses.

Dependencies: `pip install -r notebooks/requirements.txt`. Outputs are local working files and include the original license columns; they are not linked from the public site.

In [1]:
from pathlib import Path
import sys, hashlib, json, platform
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'raw_data').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'raw_data').is_dir(), 'Run from the repository or notebooks directory'
sys.path.insert(0, str(ROOT))
from processing.address_normalization import normalize_address, VERSION
INPUT = ROOT / 'raw_data' / 'Licensed Short-Term Rental Properties (1).xlsx'
SHEET = 'Sheet'
ADDRESS_COLUMN = 'Address'
UNIT_COLUMN = 'Unit Number'
OUTPUT = ROOT / 'processed_data' / 'licenses'
OUTPUT.mkdir(parents=True, exist_ok=True)
print(INPUT)

C:\Users\Admin\savannah-stvr-cad-study\raw_data\Licensed Short-Term Rental Properties (1).xlsx


In [2]:
source_hash = hashlib.sha256(INPUT.read_bytes()).hexdigest()
source = pd.read_excel(INPUT, sheet_name=SHEET, dtype=str).fillna('')
assert {ADDRESS_COLUMN, UNIT_COLUMN, 'Permit Number', 'APN'}.issubset(source.columns)
print(f'{len(source):,} license rows loaded; APNs and permits read as strings.')
display(source[[ADDRESS_COLUMN, UNIT_COLUMN, 'Permit Number', 'APN']].head())

1,683 license rows loaded; APNs and permits read as strings.


,Address,Unit Number,Permit Number,APN
0,501 EAST MCDONOUGH ST SAVANNAH GA 31401,A,SVR-00001,2001408010
1,101 EAST GASTON ST SAVANNAH GA 31401,14-A,SVR-01006,2003260030
2,110 EAST OGLETHORPE AVE SAVANNAH GA 31401,B,SVR-01008,2001507010
3,537 EAST CHARLTON ST SAVANNAH GA 31401,,SVR-01009,2001419055
4,530 EAST BROUGHTON ST SAVANNAH GA 31401,,SVR-01018,2000515007


## Check the normalization rules
These checks cover directionals, street names, embedded units, fractions, unit conflicts, and alternate-address notes. They do not certify delivery validity.

In [3]:
a = normalize_address('501 EAST MCDONOUGH STREET, SAVANNAH, GA, 31401', 'A')
assert a['normalized_delivery_address'] == '501 E MCDONOUGH ST # A'
assert normalize_address('101 E GASTON ST 8B, SAVANNAH, GA, 31401')['unit_identifier'] == '8B'
assert normalize_address('101 E GASTON ST 6A, SAVANNAH, GA, 31401', '6-A')['review_required']
assert normalize_address('12 1/2 EAST BROAD STREET SAVANNAH GA 31401')['normalized_building_address'] == '12 1/2 E BROAD ST'
assert normalize_address('12 EAST BROAD STREET SAVANNAH GA 31401', 'APT 2')['unit_designator'] == 'APT'
assert normalize_address('12 W JONES ST SAVANNAH GA 31401', 'Carriage House, 14 W Jones LN at')['review_required']
assert normalize_address('12 EAST UNKNOWNZZ SAVANNAH GA 31401')['building_match_key'] == ''
assert normalize_address('')['review_required']
print('Normalization rule checks passed.')

Normalization rule checks passed.


In [4]:
normalized = pd.DataFrame([normalize_address(a, u) for a, u in zip(source[ADDRESS_COLUMN], source[UNIT_COLUMN])])
result = pd.concat([source.reset_index(drop=True), normalized], axis=1)
result.insert(0, 'source_excel_row', range(2, len(result) + 2))
result.insert(0, 'source_sheet', SHEET)
result.insert(0, 'source_file', INPUT.name)
result['source_apn_key'] = source['APN'].str.replace(r'\s+', '', regex=True)
review = result.loc[result.review_required].copy()
duplicates = result.loc[result.normalized_full_address.ne('') & result.normalized_full_address.duplicated(keep=False)].copy()
assert len(result) == len(source)
assert result['Address'].equals(source['Address'])
assert result['APN'].equals(source['APN'])
assert not result.usps_delivery_validated.any()
print(f'{len(result):,} rows; {len(review):,} need review; {len(duplicates):,} rows share normalized addresses.')
display(result[['Address','Unit Number','normalized_delivery_address','review_reasons']].head(12))
display(review[['source_excel_row','Address','Unit Number','review_reasons']].head(20))

1,683 rows; 146 need review; 60 rows share normalized addresses.


,Address,Unit Number,normalized_delivery_address,review_reasons
0,501 EAST MCDONOUGH ST SAVANNAH GA 31401,A,501 E MCDONOUGH ST # A,
1,101 EAST GASTON ST SAVANNAH GA 31401,14-A,101 E GASTON ST # 14-A,
2,110 EAST OGLETHORPE AVE SAVANNAH GA 31401,B,110 E OGLETHORPE AVE # B,
3,537 EAST CHARLTON ST SAVANNAH GA 31401,,537 E CHARLTON ST,
4,530 EAST BROUGHTON ST SAVANNAH GA 31401,,530 E BROUGHTON ST,
5,532 EAST BROUGHTON ST SAVANNAH GA 31401,,532 E BROUGHTON ST,
6,115 WEST HUNTINGDON ST SAVANNAH GA 31401,,115 W HUNTINGDON ST,
7,516 TATTNALL ST SAVANNAH GA 31401,,516 TATTNALL ST,
8,422 EAST STATE ST SAVANNAH GA 31401,,422 E STATE ST,
9,326 EAST HARRIS ST SAVANNAH GA 31401,,326 E HARRIS ST,


,source_excel_row,Address,Unit Number,review_reasons
16,18,424 EAST YORK LN SAVANNAH GA 31401,Carriage House,unit_note_or_alternate_address_requires_review
19,21,"101 E GASTON ST 6A, SAVANNAH, GA, 31401",6-A,conflicting_unit_fields
22,24,"101 E GASTON ST 4A, SAVANNAH, GA, 31401",4-A,conflicting_unit_fields
53,55,204 WEST JONES ST SAVANNAH GA 31401,GARDEN,unit_note_or_alternate_address_requires_review
61,63,505 EAST JONES ST SAVANNAH GA 31401,GARDEN,unit_note_or_alternate_address_requires_review
63,65,301 EAST CHARLTON ST SAVANNAH GA 31401,Carriage House,unit_note_or_alternate_address_requires_review
75,77,219 EAST GASTON ST SAVANNAH GA 31401,Garden,unit_note_or_alternate_address_requires_review
141,143,501 EAST WALDBURG ST SAVANNAH GA 31401,Main House,unit_note_or_alternate_address_requires_review
149,151,120 EAST CHARLTON LN SAVANNAH GA 31401,Carriage House,unit_note_or_alternate_address_requires_review
157,159,28 EAST JONES LN SAVANNAH GA 31401,Carriage House,unit_note_or_alternate_address_requires_review


In [5]:
result.to_csv(OUTPUT / 'licenses_normalized.csv', index=False)
review.to_csv(OUTPUT / 'licenses_address_review.csv', index=False)
duplicates.to_csv(OUTPUT / 'licenses_duplicate_addresses.csv', index=False)
summary = {
    'input_file': str(INPUT.relative_to(ROOT)), 'input_sha256': source_hash,
    'source_sheet': SHEET, 'normalization_version': VERSION,
    'normalizer_sha256': hashlib.sha256((ROOT / 'processing/address_normalization.py').read_bytes()).hexdigest(),
    'executed_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(), 'pandas_version': pd.__version__,
    'rows': len(result), 'review_rows': len(review), 'duplicate_address_rows': len(duplicates),
    'review_reason_counts': review.review_reasons.str.split('; ').explode().value_counts().to_dict(),
    'usps_delivery_validated': False,
    'note': 'No rows dropped; no parcel/CAD matching; original source columns preserved.'
}
(OUTPUT / 'normalization_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
assert hashlib.sha256(INPUT.read_bytes()).hexdigest() == source_hash, 'Input changed'
display(summary)
print('Outputs:', OUTPUT)

{'input_file': 'raw_data\\Licensed Short-Term Rental Properties (1).xlsx',
 'input_sha256': 'd0a86d84be4ac6c1a3213610759ed38a5a22cb2bc134fc48966f75422b9e1790',
 'source_sheet': 'Sheet',
 'normalization_version': '1.0.0',
 'normalizer_sha256': 'a8008437ba7dcb8171b96f0a759542524cb4079c408cdbe61063fa115a85e546',
 'executed_utc': '2026-10-27T22:01:22.271674+00:00',
 'python_version': '3.14.6',
 'pandas_version': '3.0.3',
 'rows': 1683,
 'review_rows': 146,
 'duplicate_address_rows': 60,
 'review_reason_counts': {'unit_note_or_alternate_address_requires_review': 141,
  'conflicting_unit_fields': 4,
  'unparsed_or_missing_city_state_zip': 1,
  'unparsed_embedded_unit': 1},
 'usps_delivery_validated': False,
 'note': 'No rows dropped; no parcel/CAD matching; original source columns preserved.'}

Outputs: C:\Users\Admin\savannah-stvr-cad-study\processed_data\licenses


## Next stage
Review `licenses_address_review.csv` before accepting unit-level joins. Preserve `source_excel_row`, permit, original APN and original address in future joins. The source APN key is whitespace-stripped only, not a verified parcel ID. Building keys can generate parcel candidates; duplicate keys must not imply a one-to-one join. A later notebook can compare source APNs and normalized situs addresses to the parcel digest, then assess CAD address and coordinate agreement, retaining conflicts and unmatched calls.